In [1]:
import os

In [2]:
%pwd

'/Users/harshpatel/Desktop/Projects/End-to-End-Kidney-Disease-Classification-Deep-Learning-Project/notebooks'

In [3]:
os.chdir('../')

In [4]:
%pwd

'/Users/harshpatel/Desktop/Projects/End-to-End-Kidney-Disease-Classification-Deep-Learning-Project'

In [5]:
import dagshub

dagshub.init(
    repo_owner='harshpatel16052005',
    repo_name='End-to-End-Kidney-Disease-Classification-Deep-Learning-Project',
    mlflow=True
)

Accessing as harshpatel16052005

Initialized MLflow to track repo 
"harshpatel16052005/End-to-End-Kidney-Disease-Classification-Deep-Learning-Project"

Repository harshpatel16052005/End-to-End-Kidney-Disease-Classification-Deep-Learning-Project initialized!

In [6]:
import tensorflow as tf

In [7]:
model = tf.keras.models.load_model("artifacts/training/model.h5")

2026-06-12 14:05:57.888351: I metal_plugin/src/device/metal_device.cc:1154] Metal device set to: Apple M2
2026-06-12 14:05:57.888391: I metal_plugin/src/device/metal_device.cc:296] systemMemory: 8.00 GB
2026-06-12 14:05:57.888400: I metal_plugin/src/device/metal_device.cc:313] maxCacheSize: 2.67 GB
2026-06-12 14:05:57.888724: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:306] Could not identify NUMA node of platform GPU ID 0, defaulting to 0. Your kernel may not have been built with NUMA support.
2026-06-12 14:05:57.889201: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:272] Created TensorFlow device (/job:localhost/replica:0/task:0/device:GPU:0 with 0 MB memory) -> physical PluggableDevice (device: 0, name: METAL, pci bus id: <undefined>)


In [8]:
from dataclasses import dataclass
from pathlib import Path

@dataclass(frozen=True)
class EvaluationConfig:
    path_of_model: Path
    training_data: Path
    all_params: dict
    mlflow_uri: str
    params_image_size: list
    params_batch_size: int


In [9]:
from KidneyDiseaseClassification.constants import *
from KidneyDiseaseClassification.utils.common import read_yaml,create_directories,save_json

In [10]:
class ConfigurationManager:
    def __init__(
        self,
        config_filepath = CONFIG_FILE_PATH,
        params_filepath = PARAMS_FILE_PATH):
        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)
        create_directories([self.config.artifacts_root])

    def get_evaluation_config(self)->EvaluationConfig:
        eval_config = EvaluationConfig(
            path_of_model="artifacts/training/model.h5",
            training_data="artifacts/data_ingestion/kidney-ct-scan-image",
            all_params=self.params,
            mlflow_uri="https://dagshub.com/harshpatel16052005/End-to-End-Kidney-Disease-Classification-Deep-Learning-Project.mlflow",
            params_image_size = self.params.IMAGE_SIZE,
            params_batch_size=self.params.BATCH_SIZE
    )

        return eval_config

        

In [ ]:
from urllib.parse import urlparse
from pathlib import Path
import tensorflow as tf
import mlflow

In [12]:
print("Tracking URI:", mlflow.get_tracking_uri())
print("Registry URI:", mlflow.get_registry_uri())

Tracking URI: https://dagshub.com/harshpatel16052005/End-to-End-Kidney-Disease-Classification-Deep-Learning-Project.mlflow
Registry URI: https://dagshub.com/harshpatel16052005/End-to-End-Kidney-Disease-Classification-Deep-Learning-Project.mlflow


In [ ]:
class Evaluation:
    def __init__(self,config:ConfigurationManager):
        self.config = config

    def _valid_generator(self):

        datagenerator_kwargs = dict(
            rescale = 1./255,
            validation_split = 0.30
        )

        dataflow_kwargs = dict(
            target_size = self.config.params_image_size[:-1],
            batch_size = self.config.params_batch_size,
            interpolation = 'bilinear'
        )

        valid_datagenerator = tf.keras.preprocessing.image.ImageDataGenerator(
            **datagenerator_kwargs
        )

        self.valid_generator = valid_datagenerator.flow_from_directory(
            directory = self.config.training_data,
            subset = "validation",
            shuffle = False,
            **dataflow_kwargs
        )
    
    @staticmethod
    def load_model(path:Path)->tf.keras.Model:
        return tf.keras.models.load_model(path)

    def evaluation(self):
        self.model = self.load_model(self.config.path_of_model)
        self._valid_generator()
        self.score = model.evaluate(self.valid_generator)
        self.save_score()

    def save_score(self):
        scores = {"loss":self.score[0] , "accuracy":self.score[1]}
        save_json(path=Path("scores.json") , data=scores)

    def log_into_mlflow(self):
        mlflow.set_tracking_uri(self.config.mlflow_uri)
        mlflow.set_registry_uri(self.config.mlflow_uri)
        mlflow.set_experiment("kidney-disease-classification")  
        tracking_url_type_store = urlparse(mlflow.get_tracking_uri()).scheme
        
        with mlflow.start_run():
            mlflow.log_params(self.config.all_params)
            mlflow.log_metrics(
                {"loss": self.score[0], "accuracy": self.score[1]}
            )
            # Model registry does not work with file store
            if tracking_url_type_store != "file":

                # Register the model
                # There are other ways to use the Model Registry, which depends on the use case,
                # please refer to the doc for more information:
                # https://mlflow.org/docs/latest/model-registry.html#api-workflow
                mlflow.keras.log_model(self.model, "model", registered_model_name="VGG16Model")
            else:
                mlflow.keras.log_model(self.model, "model")


In [14]:
from KidneyDiseaseClassification.utils.logger import logger
from KidneyDiseaseClassification.utils.exception import CustomException
import sys

In [ ]:
try:
    config  = ConfigurationManager()
    eval_config = config.get_evaluation_config()
    evaluation  = Evaluation(eval_config)
    evaluation.evaluation()
    evaluation.log_into_mlflow()
except Exception as e:
    logger.exception(e)
    raise CustomException(e,sys)

ERROR:KidneyDiseaseClassifierLogger:name 'ConfigurationManage' is not defined
Traceback (most recent call last):
  File "/var/folders/d6/_yc2hmzn3rzgpnz6c42v07_r0000gn/T/ipykernel_14707/691689106.py", line 2, in <module>
    config  = ConfigurationManage()
NameError: name 'ConfigurationManage' is not defined. Did you mean: 'ConfigurationManager'?


CustomException: 
==================================================
File: /var/folders/d6/_yc2hmzn3rzgpnz6c42v07_r0000gn/T/ipykernel_14707/691689106.py
Line: 2
Error: name 'ConfigurationManage' is not defined
==================================================

In [21]:
import mlflow

exp = mlflow.get_experiment_by_name(
    "kidney-disease-classification"
)

print(exp.experiment_id)

2


In [23]:
import mlflow

exp = mlflow.get_experiment_by_name(
    "kidney-disease-classification"
)

print(exp.experiment_id)

2


In [24]:
client = mlflow.tracking.MlflowClient()

client.delete_run("RUN_ID")

RestException: RESOURCE_DOES_NOT_EXIST: Run not found